In [10]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

df = pd.read_csv("../data/processed/features_v2.csv", parse_dates=["date"])
df["target"] = np.log1p(df["market_value_in_eur"])  # predicting log value because raw values are heavily skewed

# Splitting by time instead of randomly: the model learns from the past and is judged on the future
train = df[df["date"] < "2024-01-01"]
valid = df[(df["date"] >= "2024-01-01") & (df["date"] < "2025-01-01")]
test = df[df["date"] >= "2025-01-01"]  # not touching this until the final model is chosen
print(len(train), len(valid), len(test))

442052 45368 66304


In [11]:
num_cols = [
    "age", "age_sq", "height_in_cm", "has_stats", "league_covered",
    "games_12m", "goals_12m", "assists_12m", "minutes_played_12m",
    "yellow_cards_12m", "red_cards_12m", "goals_per90", "assists_per90",
    "club_n_prev", "club_avg_lv_prev", "league_avg_lv_prev",
]
cat_cols = ["position", "sub_position", "foot", "country_of_citizenship",
            "player_club_domestic_competition_id"]
features = num_cols + cat_cols

In [12]:
results = []  # collecting every model's scores here so we can compare them later

def evaluate(name, model, data, split, X=None):
    pred_log = model.predict(data[features] if X is None else X)
    actual, pred = np.expm1(data["target"]), np.expm1(pred_log)  # converting back to euros for readable errors
    row = {
        "model": name, "split": split,
        "rmse_log": np.sqrt(mean_squared_error(data["target"], pred_log)),
        "r2_log": r2_score(data["target"], pred_log),
        "mae_eur": mean_absolute_error(actual, pred),
        "median_pct_err": (np.abs(pred - actual) / actual).median() * 100,
    }
    results.append(row)
    return pd.DataFrame([row]).round(3)

In [13]:
# Baseline 0: always predicting the training median, to show what "no skill" scores
dummy = DummyRegressor(strategy="median").fit(train[features], train["target"])
print(evaluate("dummy_median", dummy, valid, "valid"))

# Baseline 1: Ridge regression, with preprocessing bundled so valid/test get identical treatment
prep = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")),   # NaNs from uncovered leagues get filled; has_stats keeps that visible
                      ("scale", StandardScaler())]), num_cols),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="constant", fill_value="unknown")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=200))]), cat_cols),
])
ridge = Pipeline([("prep", prep), ("model", Ridge(alpha=10))])
ridge.fit(train[features], train["target"])
print(evaluate("ridge", ridge, valid, "valid"))

          model  split  rmse_log  r2_log      mae_eur  median_pct_err
0  dummy_median  valid     1.662  -0.151  2652320.909          73.333
   model  split  rmse_log  r2_log      mae_eur  median_pct_err
0  ridge  valid       0.9   0.662  1616461.592          50.932


In [14]:
# Trees handle NaN and categories natively, so no scaling or one-hot encoding is needed here
cat_types = {c: pd.CategoricalDtype(sorted(train[c].dropna().unique())) for c in cat_cols}

def to_tree_frame(d):
    X = d[features].copy()
    for c in cat_cols:
        X[c] = X[c].astype(cat_types[c])  # categories unseen in training become NaN instead of crashing
    return X

Xtr, Xva = to_tree_frame(train), to_tree_frame(valid)
ytr, yva = train["target"], valid["target"]

In [15]:
from xgboost import XGBRegressor

xgb = XGBRegressor(
    n_estimators=2000, learning_rate=0.05, max_depth=7,
    subsample=0.8, colsample_bytree=0.8,
    tree_method="hist", enable_categorical=True,
    early_stopping_rounds=50,  # stopping once validation error hasn't improved for 50 rounds
    n_jobs=-1, random_state=42,
)
xgb.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=200)
print("Best number of trees:", xgb.best_iteration)
print(evaluate("xgboost", xgb, valid, "valid", X=Xva))

[0]	validation_0-rmse:1.53245
[200]	validation_0-rmse:0.77898
[400]	validation_0-rmse:0.76538
[600]	validation_0-rmse:0.75909
[800]	validation_0-rmse:0.75634
[1000]	validation_0-rmse:0.75457
[1197]	validation_0-rmse:0.75370
Best number of trees: 1147
     model  split  rmse_log  r2_log      mae_eur  median_pct_err
0  xgboost  valid     0.754   0.763  1302660.235          42.383


In [16]:
from lightgbm import LGBMRegressor, early_stopping, log_evaluation

lgbm = LGBMRegressor(
    n_estimators=2000, learning_rate=0.05, num_leaves=63,
    subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
    n_jobs=-1, random_state=42, verbose=-1,
)
lgbm.fit(
    Xtr, ytr, eval_set=[(Xva, yva)],
    callbacks=[early_stopping(50), log_evaluation(200)],  # categorical columns are detected from the category dtype
)
print(evaluate("lightgbm", lgbm, valid, "valid", X=Xva))

d:\football-value-model\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 50 rounds
[200]	valid_0's l2: 0.603603
[400]	valid_0's l2: 0.589222
[600]	valid_0's l2: 0.582301
[800]	valid_0's l2: 0.577546
[1000]	valid_0's l2: 0.572147
[1200]	valid_0's l2: 0.568529
[1400]	valid_0's l2: 0.565511
[1600]	valid_0's l2: 0.562988
[1800]	valid_0's l2: 0.560895
[2000]	valid_0's l2: 0.558889
Did not meet early stopping. Best iteration is:
[1995]	valid_0's l2: 0.558873
      model  split  rmse_log  r2_log      mae_eur  median_pct_err
0  lightgbm  valid     0.748   0.767  1280962.083          42.073


In [17]:
# Side-by-side comparison of everything evaluated so far
# (results keeps growing, so re-running an evaluate cell adds duplicate rows)
print(pd.DataFrame(results).round(3).to_string(index=False))

# Mean error in log space: negative means the model is underpredicting on average (the inflation effect)
for name, m, X in [("ridge", ridge, valid[features]), ("xgboost", xgb, Xva), ("lightgbm", lgbm, Xva)]:
    print(name, "mean log error:", round((m.predict(X) - yva).mean(), 3))

# Scoring XGBoost separately for leagues with performance data and the rest
for flag, label in [(1, "covered leagues"), (0, "other leagues")]:
    mask = (valid["league_covered"] == flag).to_numpy()
    print(label, "rows:", mask.sum())
    for name, m in [("xgboost", xgb), ("lightgbm", lgbm)]:
        print(evaluate(f"{name} ({label})", m, valid[mask], "valid", X=Xva[mask]))

       model split  rmse_log  r2_log     mae_eur  median_pct_err
dummy_median valid     1.662  -0.151 2652320.909          73.333
       ridge valid     0.900   0.662 1616461.592          50.932
     xgboost valid     0.754   0.763 1302660.235          42.383
    lightgbm valid     0.748   0.767 1280962.083          42.073
ridge mean log error: -0.111
xgboost mean log error: -0.017
lightgbm mean log error: -0.015
covered leagues rows: 19375
                       model  split  rmse_log  r2_log      mae_eur  \
0  xgboost (covered leagues)  valid      0.71    0.82  2314724.918   

   median_pct_err  
0          40.529  
                        model  split  rmse_log  r2_log      mae_eur  \
0  lightgbm (covered leagues)  valid     0.704   0.823  2266584.701   

   median_pct_err  
0          40.084  
other leagues rows: 25993
                     model  split  rmse_log  r2_log     mae_eur  \
0  xgboost (other leagues)  valid     0.785   0.594  548274.315   

   median_pct_err  
0         